In [2]:
# Import Libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Read Datasets
app = pd.read_csv("application_record.csv.gz", compression="gzip")
credit = pd.read_csv("credit_record.csv")

# Convert STATUS into Binary
def to_binary(status):
    if status in ['0', 'X', 'C']:
        return 1
    else:
        return 0

credit['STATUS_BIN'] = credit['STATUS'].apply(to_binary)

# Keep One Record per Customer
credit = credit.groupby('ID')['STATUS_BIN'].max().reset_index()

# Merge Datasets
final_df = pd.merge(app, credit, on='ID', how='left')

# Fill Missing Target Values
final_df['STATUS_BIN'] = final_df['STATUS_BIN'].fillna(0)

# Drop Unnecessary Column
if 'OCCUPATION_TYPE' in final_df.columns:
    final_df.drop('OCCUPATION_TYPE', axis=1, inplace=True)

# Encode Categorical Columns
le = LabelEncoder()

categorical_cols = [
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE'
]

for col in categorical_cols:
    final_df[col] = le.fit_transform(final_df[col])

# Features and Target
X = final_df.drop(['ID','STATUS_BIN'], axis=1)
y = final_df['STATUS_BIN']
print("Class Distribution:")
print(y.value_counts())

print("\nPercentage:")
print(y.value_counts(normalize=True) * 100)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train Logistic Regression Model
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr_model.fit(X_train, y_train)

# Prediction
y_pred = lr_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy Score")
print(accuracy)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(cm)

# Classification Report
print("\nClassification Report")
print(classification_report(y_test, y_pred))

Class Distribution:
0.0    402101
1.0     36456
Name: STATUS_BIN, dtype: int64

Percentage:
0.0    91.687284
1.0     8.312716
Name: STATUS_BIN, dtype: float64
Accuracy Score
0.3955331083546151

Confusion Matrix
[[30158 50263]
 [ 2756  4535]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.92      0.38      0.53     80421
         1.0       0.08      0.62      0.15      7291

    accuracy                           0.40     87712
   macro avg       0.50      0.50      0.34     87712
weighted avg       0.85      0.40      0.50     87712

